## 다중선형 회귀 문제 - 경사하강법 구현( GD )
```txt
문제
예시 데이터
근속년수, 자격증수 -> 연동. 5개 샘플 데이터 생성하기
특성 2개
```

STEP#1 - 라이브러리 임포트

In [2]:
import numpy as np

STEP#2 - 예시 데이터 샡성하기 - 넘파이 배열<br>
X: 입력값 행렬
- 행( row ) 1개 = 샘플 1명
- 열( column ) 1개 - 특성( Feature ) 1개

In [8]:
# 근속년수, 자격증 수
X = np.array([
  [1,0],
  [2,1],
  [3,1],
  [4,2],
  [5,2],
  ])
# 연봉 - 5개
y = np.array([2000, 2600, 3000, 3600, 4000], dtype=float)

In [9]:
print(type(X))

<class 'numpy.ndarray'>


In [10]:
y

array([2000., 2600., 3000., 3600., 4000.])

STEP#3 - 하이퍼파라미터 설정

In [15]:
# 학습율
lr = 0.01 # learning rate: 한번 업데이트 할때 얼마나 크게 움질일지 ( 학습률 )
epochs = 5000 # 학습 반복 횟수

In [11]:
n, d = X.shape
n

5

In [13]:
# W: 가중치 ( 특성별 영햑력 ) 벡터, 특성 2개 니까 w 도 2개로 함
W = np.zeros(d) # [0, 0] 에서 시작
# b: 절편( 기본 연봉 역할 - 0에서 시작 )
b = 0.0

경사하강법 루프

In [16]:
for epoch in range(1, epochs + 1):

    # (1) 예측값 계산
    # y_hat = X @ W + b
    # - X @ W : (n,d) @ (d,) => (n,)  (각 샘플의 선형결합 결과)
    # - + b   : 모든 샘플에 같은 절편을 더함(브로드캐스팅)
    y_hat = X @ W + b

    # (2) 오차 계산
    # error[i] = (예측 - 실제)
    error = y_hat - y

    # (3) 손실(loss) 계산: 평균제곱오차(MSE)
    # loss = mean(error^2)
    # - 값이 작아질수록 예측이 실제에 가까움
    loss = np.mean(error ** 2)

    # (4) 기울기(gradient) 계산
    # MSE = (1/n) * Σ (y_hat - y)^2
    # y_hat = XW + b 이므로,
    # dW = (2/n) * X^T @ error
    # db = (2/n) * Σ error
    # - X.T @ error : 각 특성별로 "오차와 입력의 상관"을 합산한 것(방향/크기)
    dW = (2 / n) * (X.T @ error)
    db = (2 / n) * np.sum(error)

    # (5) 경사하강 업데이트
    # - loss를 줄이려면 기울기(증가 방향) 반대로 이동해야 함
    W -= lr * dW
    b -= lr * db

    # (6) 중간 로그 출력: 500번마다 현재 손실/파라미터 확인
    if epoch % 500 == 0:
        print(f"epoch={epoch:4d} loss={loss:.2f} W={W} b={b:.2f}")

epoch= 500 loss=11855.55 W=[573.84692737 -26.64484072] b=1313.18
epoch=1000 loss=2545.74 W=[507.25803701  13.64881438] b=1494.07
epoch=1500 loss=1038.81 W=[469.10376054  73.56158885] b=1541.49
epoch=2000 loss=440.29 W=[444.92338935 116.96705561] b=1563.24
epoch=2500 loss=186.90 W=[429.25677434 145.8151188 ] b=1576.23
epoch=3000 loss=79.34 W=[419.06060291 164.68475068] b=1584.53
epoch=3500 loss=33.68 W=[412.41875238 176.9888532 ] b=1589.93
epoch=4000 loss=14.30 W=[408.09143419 185.00685624] b=1593.44
epoch=4500 loss=6.07 W=[405.2719864  190.23116318] b=1595.72
epoch=5000 loss=2.58 W=[403.4349728  193.63509171] b=1597.21


---

```txt
1. epoch 변화
  11855 -> 2.58 ( 손실율 감소됨 ) : 경사하강이 잘 동작해 MSE가 줄어듬
  W, b: 초반에 크게 움직이다가 후반으로 갈 수록 안정적으로 수렴

2. 최종 계수 해석
  W = [403, 193]( 근속연수 계수, 자격증수 계수)
  - 근속년수 1년증가 -> 연봉 약 403만원 증가함.
  - 자격증 1개 증가 -> 연봉 약 193만원 증가함
  b = 1597 -> 근속 0년, 자격증 0개 일때 기준 연봉 ( 만원 )

3. 예측식
  예측값 = 403.43*근속연수 + 193.64*자격증수 + 1597.21

  경사하강법으로 W, b가 수렴했고, 학습 데이터 기준으로 근속년수/자격증수가 연봉에
  양의 영향을 준다는 관계가 계수로 확인됨.
```

학습결과 확인

In [17]:
print('최종 W: ', W)  # 학습된 가중치 ( 근속년수 / 자격증수 가 연봉에 미치는 영향)
print('최종 b: ', b)  # 학습된 절편 ( 특성 0 일때 기본값 얼마냐? )

최종 W:  [403.4349728  193.63509171]
최종 b:  1597.2136559720498


근속 연수 6년, 자격증 3개인 사람의 연봉 예측

In [18]:
x_new = np.array([6, 3], dtype=float)
print('입력: ', x_new, '예측: ', x_new @ W+b)

입력:  [6. 3.] 예측:  4598.728767896031
